**Imports, paths, helpers, and wide power spectrum for one TIC**

In [ ]:
import os
import numpy as np
import pandas as pd
import lightkurve as lk
from astropy import units as u
import matplotlib.pyplot as plt

from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, Slider, CustomJS, Title, LinearColorMapper
from bokeh.layouts import column
from bokeh.io import output_notebook
from bokeh.resources import INLINE
from IPython.display import Image, display

output_notebook(resources=INLINE, hide_banner=True)

MASTER_CSV = "master_all_clean_best_radius.csv"
BASE_OUTDIR = "/Users/carlimankowski/research/dnuvisuals/visuals"

# helpers 
def _strip_tic_prefix(s):
    s = str(s).strip()
    return s.split()[-1] if s.upper().startswith("TIC") else s

def _load_numax_from_master(tic_id, master_csv=MASTER_CSV):
    tic = int(_strip_tic_prefix(tic_id))
    m = pd.read_csv(master_csv)
    row = m.loc[m["TICID"] == tic]
    if row.empty:
        raise ValueError(f"TIC {tic} not found in {master_csv}")
    return float(row["numax"].iloc[0])

def _nyquist_uHz(lc):
    dt_sec = np.nanmedian(np.diff(lc.time.value)) * 86400.0
    return 1e6 / (2.0 * dt_sec)  # μHz

def _fetch_lc(tic_id, authors=("SPOC","QLP"), window=299, combine=True):
    tic_id = _strip_tic_prefix(tic_id)
    search_result = None
    for auth in authors:
        sr = lk.search_lightcurve(f"TIC {tic_id}", author=auth)
        print(f"Author={auth}, found {len(sr)} products")
        if len(sr) > 0 and search_result is None:
            search_result = sr
            print(f"  Using author={auth}")
    if search_result is None:
        raise RuntimeError(f"No lightcurves found for TIC {tic_id}")

    lc_collect_og = search_result.download_all()
    lc_collect = []
    for lc in lc_collect_og:
        lc_new = lc.normalize().flatten(window_length=window)
        lc_new = lc_new.remove_outliers(sigma=2.75)
        lc_collect.append(lc_new)

    lc_collected = lk.LightCurveCollection(lc_collect)
    lc = lc_collected.stitch().normalize()

    times = lc.time.value
    fluxes = lc.flux.value
    order = np.argsort(times)
    times, fluxes = times[order], fluxes[order]

    if combine:
        for i in range(1, len(times)):
            if (times[i] - times[i-1]) > 10:
                shift = times[i] - times[i-1]
                times[i:] = times[i:] - shift

    lc_clean = lk.LightCurve(time=times, flux=fluxes)

    # quick look LC
    plt.figure(figsize=(12, 2))
    plt.plot(times, fluxes, "k", lw=0.5)
    plt.title(f"TIC {tic_id} light curve (normalized, flattened)")
    plt.xlabel("Time (BTJD)")
    plt.ylabel("Flux")
    plt.show()

    return lc_clean

def _compute_wide_ps(tic_id, lc, numax_uHz, outdir="datanew", overwrite=False):
    tic_id = _strip_tic_prefix(tic_id)
    os.makedirs(outdir, exist_ok=True)

    nyq = _nyquist_uHz(lc)

    if np.isfinite(numax_uHz) and numax_uHz > 0:
        fmin = max(0.1, 0.3 * numax_uHz)
        fmax = min(0.95*nyq, 1.7 * numax_uHz)
        if fmax <= fmin:
            fmin, fmax = 0.1, 0.95*nyq
    else:
        fmin, fmax = 0.1, 0.95*nyq

    print(f"Computing power spectrum: {fmin:.2f}–{fmax:.2f} μHz (Nyquist≈{nyq:.1f} μHz)")

    ps = lc.to_periodogram(
        method="lombscargle",
        minimum_frequency=fmin,
        maximum_frequency=fmax,
        oversample_factor=10,
        freq_unit=u.uHz,
    )

    ps_out = os.path.join(outdir, f"{tic_id}_wide_PS.txt")
    if overwrite or (not os.path.exists(ps_out)):
        df = pd.DataFrame({"freq": ps.frequency.to(u.uHz).value,
                           "power": ps.power.value})
        df.to_csv(ps_out, sep="\t", header=False, index=False)
        print(f"Saved PS to {ps_out}")
    else:
        print(f"PS already exists at {ps_out} (overwrite=False)")

    return ps

def _load_wide_ps(tic_id, outdir="datanew"):
    tic = _strip_tic_prefix(tic_id)
    fp = os.path.join(outdir, f"{tic}_wide_PS.txt")
    if not os.path.exists(fp):
        raise FileNotFoundError(f"{fp} not found — run the load cell first")
    d = pd.read_csv(fp, sep="\t", names=["freq","power"])
    return d["freq"].to_numpy(), d["power"].to_numpy(), fp

# prompt for TIC, then build LC + PS 
TIC_ID = int(input("Enter TIC ID: ").strip())
numax_val = _load_numax_from_master(TIC_ID)
print(f"TIC {TIC_ID}: numax = {numax_val:.3f} μHz")

lc = _fetch_lc(TIC_ID)
_ = _compute_wide_ps(TIC_ID, lc, numax_val, outdir="datanew", overwrite=False)


**Interactive 2D echelle with Δν slider**

In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, Slider, CustomJS, Title, LinearColorMapper
from bokeh.layouts import column

LAST_DNU = None  # will be updated each time you run an interactive cell

def _make_initial_img(freq, power, dnu, nbins_x=200, n_keep=4):
    f0 = float(freq.min())
    x  = np.mod(freq - f0, dnu)
    y  = np.floor((freq - f0) / dnu).astype(int)

    y0 = int(y.min())
    y1 = y0 + n_keep - 1

    x_edges = np.linspace(0.0, dnu, nbins_x + 1)
    img = np.zeros((n_keep, nbins_x), dtype=float)
    counts = np.zeros_like(img)

    for j, order in enumerate(range(y0, y1 + 1)):
        mask_j = (y == order)
        if np.any(mask_j):
            xj = x[mask_j]; pj = power[mask_j]
            bins = np.clip(np.digitize(xj, x_edges)-1, 0, nbins_x-1)
            sums  = np.bincount(bins, weights=pj, minlength=nbins_x)
            nbins = np.bincount(bins, minlength=nbins_x)
            img[j,:] = sums
            counts[j,:] = nbins

    nz = counts > 0
    img[nz] = img[nz] / counts[nz]

    return img, dict(f0=f0, y0=y0, y1=y1)

def interactive_echelle_js(tic_id):
    global LAST_DNU

    tic = _strip_tic_prefix(tic_id)
    numax = _load_numax_from_master(tic)
    dnu0  = float(input(f"Enter Δν (μHz) for TIC {tic}: ").strip())
    LAST_DNU = dnu0

    freq, power, used_fp = _load_wide_ps(tic)

    nbins_x = 200
    n_keep  = 4
    img0, meta = _make_initial_img(freq, power, dnu0, nbins_x=nbins_x, n_keep=n_keep)

    raw_source = ColumnDataSource(data=dict(freq=freq.tolist(), power=power.tolist()))
    img_source = ColumnDataSource(data=dict(
        image=[img0.tolist()],
        x=[0.0],
        y=[meta["y0"]],
        dw=[dnu0],
        dh=[n_keep]
    ))

    mapper = LinearColorMapper(palette="Viridis256")

    title = Title(text=f"Echelle — TIC {tic}   Δν={dnu0:.3f} μHz   (from {os.path.basename(used_fp)})")
    p = figure(title=title,
               x_axis_label="Frequency mod Δν (μHz)",
               y_axis_label="Order index",
               width=650, height=520)
    p.image(image='image', x='x', y='y', dw='dw', dh='dh',
            source=img_source, color_mapper=mapper)

    slider = Slider(start=max(0.1, dnu0*0.7), end=dnu0*1.3,
                    value=dnu0, step=0.01, title="Δν (μHz)")

    callback = CustomJS(
        args=dict(
            raw=raw_source,
            imgsrc=img_source,
            title_obj=p.title,
            nbins_x=nbins_x,
            n_keep=n_keep,
            f0=float(meta["f0"]),
            y0=meta["y0"]
        ),
        code="""
        const freq  = raw.data['freq'];
        const power = raw.data['power'];
        const dnu   = cb_obj.value;
        const nbins = nbins_x;
        const keep  = n_keep;
        const f0loc = f0;
        const y0loc = y0;

        const img = [];
        const counts = [];
        for (let j = 0; j < keep; j++) {
            img[j] = new Array(nbins).fill(0.0);
            counts[j] = new Array(nbins).fill(0.0);
        }
        const step = dnu / nbins;

        const N = freq.length;
        for (let i = 0; i < N; i++) {
            const xi_raw = freq[i] - f0loc;
            const yi = Math.floor(xi_raw / dnu);
            const rel = yi - y0loc;
            if (rel < 0 || rel >= keep) continue;
            let xm = xi_raw % dnu;
            if (xm < 0) xm += dnu;
            let b = Math.floor(xm / step);
            if (b < 0) b = 0;
            if (b >= nbins) b = nbins - 1;
            img[rel][b] += power[i];
            counts[rel][b] += 1.0;
        }

        for (let j = 0; j < keep; j++) {
            for (let b = 0; b < nbins; b++) {
                if (counts[j][b] > 0) img[j][b] /= counts[j][b];
            }
        }

        imgsrc.data['image'] = [img];
        imgsrc.data['x'] = [0.0];
        imgsrc.data['y'] = [y0loc];
        imgsrc.data['dw'] = [dnu];
        imgsrc.data['dh'] = [keep];
        imgsrc.change.emit();

        title_obj.text = `Echelle — TIC ${"%s"}   Δν=${dnu.toFixed(3)} μHz`.replace("%s", "%s");
        """.replace("%s", str(tic))
    )

    slider.js_on_change("value", callback)
    show(column(p, slider))

# run for TIC from Cell 1
interactive_echelle_js(TIC_ID)


**Interactive collapsed echelle (4×Δν line plot) with Δν slider**

In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, Slider
from bokeh.layouts import column

def collapsed_multi(freq, power, dnu, numax=None, width=None, bins=200, n_panels=4):
    if width is None:
        width = 4 * dnu
    if numax is not None:
        mask = (freq > numax - width/2) & (freq < numax + width/2)
        freq, power = freq[mask], power[mask]

    x = (freq % dnu)
    edges = np.linspace(0, dnu, bins)
    digitized = np.digitize(x, edges)
    collapsed = [power[digitized == i].mean() if np.any(digitized == i) else 0.0
                 for i in range(1, len(edges))]
    centers = 0.5 * (edges[1:] + edges[:-1])

    x_full = np.concatenate([centers + k*dnu for k in range(n_panels)])
    y_full = np.tile(collapsed, n_panels)
    return x_full, y_full

def adjust_collapsed_multi(tic_id):
    global LAST_DNU

    name = f"TIC {_strip_tic_prefix(tic_id)}"
    numax = _load_numax_from_master(tic_id)
    dnu = float(input(f"Enter Δν (μHz) for {name}: ").strip())
    LAST_DNU = dnu

    freq, power, _ = _load_wide_ps(tic_id)

    eps = max(0.05, dnu*0.02)
    dnu_min = dnu - eps
    dnu_max = dnu + eps
    dnu0 = dnu

    bins = 200
    n_panels = 4

    x_full, y_full = collapsed_multi(freq, power, dnu0, numax=numax,
                                     bins=bins, n_panels=n_panels)
    source = ColumnDataSource(data=dict(x=x_full, y=y_full))

    p = figure(title=f"Collapsed Echelle ({n_panels}×Δν) for {name}",
               x_axis_label="Frequency mod Δν (μHz), repeated panels",
               y_axis_label="Collapsed Power")
    p.line('x', 'y', source=source)

    freq_list = freq.tolist()
    power_list = power.tolist()

    slider = Slider(start=dnu_min, end=dnu_max, value=dnu0,
                    step=0.001, title="Δν (μHz)")

    callback = CustomJS(args=dict(source=source,
                                  freq=freq_list,
                                  power=power_list,
                                  numax=numax,
                                  bins=bins,
                                  n_panels=n_panels), code="""
        const dnu = cb_obj.value
        const width = 4 * dnu
        const edges = []
        for (let i=0; i<=bins; i++) edges.push(i*dnu/bins)

        let f = [], p = []
        for (let i=0; i<freq.length; i++) {
            if (freq[i] > numax - width/2 && freq[i] < numax + width/2) {
                f.push(freq[i])
                p.push(power[i])
            }
        }

        const sums = Array(bins).fill(0)
        const counts = Array(bins).fill(0)

        for (let i=0; i<f.length; i++) {
            const mod = f[i] % dnu
            const bin = Math.floor(mod / (dnu/bins))
            if (bin >= 0 && bin < bins) {
                sums[bin] += p[i]
                counts[bin] += 1
            }
        }

        const base_x = []
        const base_y = []
        for (let i=0; i<bins; i++) {
            base_x.push((edges[i]+edges[i+1])/2)
            base_y.push(counts[i] > 0 ? sums[i]/counts[i] : 0)
        }

        let x_full = []
        let y_full = []
        for (let k=0; k<n_panels; k++) {
            for (let i=0; i<base_x.length; i++) {
                x_full.push(base_x[i] + k*dnu)
                y_full.push(base_y[i])
            }
        }

        source.data = {x: x_full, y: y_full}
        source.change.emit()
    """)

    slider.js_on_change("value", callback)
    show(column(slider, p))

#run for TIC from Cell 1 
adjust_collapsed_multi(TIC_ID)


**Interactive 1D overlay heatmap with Δν slider**

In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, Slider, Title, LinearColorMapper
from bokeh.layouts import column

def interactive_overlay_heatmap(tic_id):
    global LAST_DNU

    tic = _strip_tic_prefix(tic_id)
    numax = _load_numax_from_master(tic)
    dnu0  = float(input(f"Enter Δν (μHz) for TIC {tic}: ").strip())
    LAST_DNU = dnu0

    freq_all, power_all, used_fp = _load_wide_ps(tic)

    # limit to ±2Δν around νmax
    mask = (freq_all > numax - 2*dnu0) & (freq_all < numax + 2*dnu0)
    freq = freq_all[mask] if np.any(mask) else freq_all
    power = power_all[mask] if np.any(mask) else power_all

    nbins_x = 200

    def make_overlay(freq, power, dnu):
        f0 = float(freq.min())
        x  = np.mod(freq - f0, dnu)

        x_edges = np.linspace(0.0, dnu, nbins_x+1)
        bins = np.clip(np.digitize(x, x_edges)-1, 0, nbins_x-1)
        sums  = np.bincount(bins, weights=power, minlength=nbins_x)
        nbins = np.bincount(bins, minlength=nbins_x)
        img = np.zeros_like(sums, dtype=float)
        img[nbins > 0] = sums[nbins > 0] / nbins[nbins > 0]
        return img

    img0 = make_overlay(freq, power, dnu0).tolist()

    source = ColumnDataSource(data=dict(
        image=[[img0]],  # 1×nbins_x image
        x=[0.0],
        y=[0.0],
        dw=[dnu0],
        dh=[1.0]
    ))

    mapper = LinearColorMapper(palette="Greys256")

    title = Title(text=f"Overlay Collapsed Echelle — TIC {tic}   Δν={dnu0:.3f} μHz")
    p = figure(title=title,
               x_axis_label="Frequency mod Δν (μHz)",
               y_axis_label="",
               width=650, height=200,
               y_range=(0,1))
    p.image(image='image', x='x', y='y', dw='dw', dh='dh',
            source=source, color_mapper=mapper)

    slider = Slider(start=max(0.1, dnu0*0.7), end=dnu0*1.3,
                    value=dnu0, step=0.01, title="Δν (μHz)")

    callback = CustomJS(
        args=dict(source=source,
                  freq=freq.tolist(),
                  power=power.tolist(),
                  numax=numax,
                  nbins_x=nbins_x,
                  title_obj=p.title),
        code="""
        const dnu = cb_obj.value;
        const N = freq.length;
        const nbins = nbins_x;
        const step = dnu / nbins;

        let f0 = Math.min(...freq);
        let x = new Array(N);
        for (let i=0; i<N; i++) {
            x[i] = (freq[i] - f0) % dnu;
            if (x[i] < 0) x[i] += dnu;
        }

        let sums = new Array(nbins).fill(0.0);
        let counts = new Array(nbins).fill(0.0);

        for (let i=0; i<N; i++) {
            let b = Math.floor(x[i] / step);
            if (b < 0) b = 0;
            if (b >= nbins) b = nbins-1;
            sums[b] += power[i];
            counts[b] += 1.0;
        }

        let img = new Array(nbins).fill(0.0);
        for (let b=0; b<nbins; b++) {
            if (counts[b] > 0) img[b] = sums[b]/counts[b];
        }

        source.data['image'] = [ [img] ];
        source.data['x'] = [0.0];
        source.data['y'] = [0.0];
        source.data['dw'] = [dnu];
        source.data['dh'] = [1.0];
        source.change.emit();

        title_obj.text = `Overlay Collapsed Echelle — TIC ${"%s"}   Δν=${dnu.toFixed(3)} μHz`.replace("%s", "%s");
        """.replace("%s", str(tic))
    )

    slider.js_on_change("value", callback)
    show(column(p, slider))

#run for TIC from Cell 1 
interactive_overlay_heatmap(TIC_ID)


**Save static PNGs using the most recent LAST_DNU**

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt

def collapsed_multi_static_like_interactive(freq, power, dnu, numax, bins=200, n_panels=4):
    mask = (freq > numax - 2*dnu) & (freq < numax + 2*dnu)
    freq_use = freq[mask] if np.any(mask) else freq
    power_use = power[mask] if np.any(mask) else power

    x = (freq_use % dnu)
    edges = np.linspace(0, dnu, bins)
    digitized = np.digitize(x, edges)
    collapsed = [power_use[digitized == i].mean() if np.any(digitized == i) else 0.0
                 for i in range(1, len(edges))]
    centers = 0.5 * (edges[1:] + edges[:-1])

    x_full = np.concatenate([centers + k*dnu for k in range(n_panels)])
    y_full = np.tile(collapsed, n_panels)
    return x_full, y_full

def echelle_2d(freq, power, dnu, nbins_x=200):
    f0 = float(freq.min())
    x  = np.mod(freq - f0, dnu)
    y  = np.floor((freq - f0) / dnu).astype(int)
    y0 = int(y.min())
    y1 = int(y.max())
    n_orders = max(1, y1 - y0 + 1)

    x_edges = np.linspace(0.0, dnu, nbins_x+1)
    img     = np.zeros((n_orders, nbins_x), dtype=float)
    counts  = np.zeros_like(img)

    for j in range(n_orders):
        mask = (y == (y0 + j))
        if not np.any(mask):
            continue
        xj = x[mask]
        pj = power[mask]
        bins = np.clip(np.digitize(xj, x_edges) - 1, 0, nbins_x-1)
        sums  = np.bincount(bins, weights=pj, minlength=nbins_x)
        nbins = np.bincount(bins, minlength=nbins_x)
        img[j, :]    = sums
        counts[j, :] = nbins

    nonzero = counts > 0
    img[nonzero] = img[nonzero] / counts[nonzero]

    extent = (0.0, dnu, y0, y0 + n_orders)
    return img, extent

def overlay_flattened(freq, power, numax, dnu, bins=200):
    mask = (freq > numax - 2*dnu) & (freq < numax + 2*dnu)
    f = freq[mask] if np.any(mask) else freq
    p = power[mask] if np.any(mask) else power

    x = np.mod(f - f.min(), dnu)
    x_edges = np.linspace(0.0, dnu, bins+1)
    bins_idx = np.clip(np.digitize(x, x_edges)-1, 0, bins-1)
    sums  = np.bincount(bins_idx, weights=p, minlength=bins)
    counts = np.bincount(bins_idx, minlength=bins)
    img = np.zeros_like(sums, dtype=float)
    img[counts > 0] = sums[counts > 0] / counts[counts > 0]
    return img

def save_and_display_echelles(tic_id, dnu_val=None):
    # Use the most recent Δν if not explicitly supplied
    if dnu_val is None:
        try:
            dnu_val = float(LAST_DNU)
        except Exception:
            raise RuntimeError("No Δν set yet — run one of the interactive cells first to define LAST_DNU.")

    tic_clean = _strip_tic_prefix(tic_id)
    outdir = os.path.join(BASE_OUTDIR, tic_clean)
    os.makedirs(outdir, exist_ok=True)

    freq, power, used_fp = _load_wide_ps(tic_id)
    numax_val = _load_numax_from_master(tic_id)

    # collapsed 4× plot
    x_full, y_full = collapsed_multi_static_like_interactive(freq, power,
                                                             dnu_val, numax_val,
                                                             bins=200, n_panels=4)
    plt.figure(figsize=(10,4))
    plt.plot(x_full, y_full, lw=1)
    plt.xlabel("Frequency mod Δν (μHz), repeated panels")
    plt.ylabel("Collapsed Power")
    plt.title(f"Collapsed Echelle (4×Δν, ±2Δν window) — TIC {tic_clean}   Δν={dnu_val:.3f} μHz")
    out1 = os.path.join(outdir, f"collapsed4_TIC{tic_clean}.png")
    plt.savefig(out1, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved {out1}")

    #overlay flattened strip 
    img_flat = overlay_flattened(freq, power, numax_val, dnu_val, bins=200)
    plt.figure(figsize=(8,3))
    plt.imshow(img_flat[np.newaxis,:], aspect="auto", cmap="Greys_r",
               extent=(0, dnu_val, 0, 1), origin="lower")
    plt.xlabel("Frequency mod Δν (μHz)")
    plt.title(f"Overlay Collapsed Echelle — TIC {tic_clean}   Δν={dnu_val:.3f} μHz")
    out_flat = os.path.join(outdir, f"overlay_TIC{tic_clean}.png")
    plt.savefig(out_flat, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved {out_flat}")

    # 2D echelle
    fmin_2d = numax_val - 2*dnu_val
    fmax_2d = numax_val + 2*dnu_val
    mask2d = (freq >= fmin_2d) & (freq <= fmax_2d)
    freq_2d = freq[mask2d]
    power_2d = power[mask2d]

    img2d, extent = echelle_2d(freq_2d, power_2d, dnu_val, nbins_x=200)
    plt.figure(figsize=(7,6))
    plt.imshow(img2d, origin="lower", aspect="auto", extent=extent)
    plt.colorbar(label="Power (avg within bin)")
    plt.xlabel("Frequency mod Δν (μHz)")
    plt.ylabel("Order index")
    plt.title(f"Echelle (2D, ±2Δν around νmax) — TIC {tic_clean}   Δν={dnu_val:.3f} μHz")

    f0 = fmin_2d
    order_at_numax = int(np.floor((numax_val - f0) / dnu_val))
    plt.axhline(order_at_numax, color="red", linestyle="--", lw=1, label="νmax")
    plt.legend(loc="upper right")

    out2 = os.path.join(outdir, f"echelle4_TIC{tic_clean}.png")
    plt.savefig(out2, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved {out2}")

    # show all three
    display(Image(filename=out1))
    display(Image(filename=out_flat))
    display(Image(filename=out2))

print(f"Saving static echelles for TIC {TIC_ID} with Δν = {LAST_DNU}")
save_and_display_echelles(TIC_ID, dnu_val=LAST_DNU)
